In [3]:
"""
Pseudobulk Differential Expression Pipeline — pyDESeq2
========================================================
For each patient × region (CT, IM, Stroma):
  1. Identify DC-enriched spots (Dendritic_Cell proportion > 5%)
  2. Sum raw counts across those spots → one pseudobulk sample
  3. Run pyDESeq2 with design: ~ QC + MLH1 + age + sex + region * mets_status

Uses proper spatial boundary method for CT/IM/Stroma definition.

Run from your samples directory:
    cd /Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples
    python pseudobulk_deseq2_dc.py
"""

import scanpy as sc
import pandas as pd
import numpy as np
import os
import re
import warnings
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import KDTree
from scipy import sparse

warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
VISIUM_DIR = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"
META_PATH  = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"
OUTPUT_DIR = os.path.join(VISIUM_DIR, "pseudobulk_DC_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# PARAMETERS
# ============================================================
TARGET_CELL_TYPE = "Dendritic_Cell"  # h6 group name
DC_THRESHOLD     = 0.05             # 5% proportion threshold
MIN_SPOTS        = 20               # minimum spots to form a pseudobulk sample
MIN_GENES_EXPR   = 10               # filter genes expressed in fewer than this many samples

# ============================================================
# CELL TYPE MAPPING — your h6 mapping
# ============================================================
def map_to_h6(col):
    clean = col.replace("meanscell_abundance_w_sf_", "")
    if clean.startswith("Tumor"): return "Tumor"
    if clean.startswith("cE"): return "Epithelial_Normal"
    if clean.startswith("cS"):
        try:
            num = int(re.search(r'\d+', clean).group())
            if 1 <= num <= 14: return "Endothelial"
            if num == 32: return "Smooth_Muscle"
            return "Fibroblast_Stromal"
        except: return "Fibroblast_Stromal"
    if clean.startswith("cB"): return "B_Cell"
    if clean.startswith("cP"): return "Plasma"
    if clean.startswith("cM"):
        if "cM10" in clean: return "Neutrophil_Granulocyte"
        if "cM01" in clean or "cM02" in clean: return "Mono_Macrophage"
        return "Dendritic_Cell"
    if clean.startswith("cMA"): return "Mast"
    if clean.startswith("cTNI"):
        try:
            num = int(re.search(r'\d+', clean).group())
            if 1 <= num <= 7: return "CD4_T"
            if 8 <= num <= 9: return "Treg"
            if 10 <= num <= 16: return "CD8_T"
            if 17 <= num <= 22: return "T_Other_GammaDelta"
            if 23 <= num <= 25: return "NK"
        except: return "T_Other"
    return "Other"

# ============================================================
# REGION DEFINITION — from Define_regions.ipynb
# ============================================================
def define_regions_spatial(adata, pixel_scale_100um):
    coords = adata.obsm["spatial"]

    tumor_labels  = ["cancer_cancer"]
    stroma_labels = ["benign_stroma", "benign_fat", "inter", "stroma"]

    histo = adata.obs["histology"].astype(str)
    is_tumor  = histo.isin(tumor_labels).values
    is_stroma = histo.isin(stroma_labels).values

    region_labels = np.array(["Other"] * adata.n_obs, dtype=object)

    if is_tumor.sum() == 0:
        region_labels[is_stroma] = "Stroma"
        return region_labels
    if is_stroma.sum() == 0:
        region_labels[is_tumor] = "CT"
        return region_labels

    # STEP 1: Find biological interface
    tumor_coords = coords[is_tumor]
    stroma_tree  = KDTree(coords[is_stroma])
    dists_to_stroma, _ = stroma_tree.query(tumor_coords, k=1)

    neighbor_threshold = 1.5 * pixel_scale_100um
    boundary_subset_mask = dists_to_stroma <= neighbor_threshold

    tumor_indices = np.where(is_tumor)[0]
    true_boundary_indices = tumor_indices[boundary_subset_mask]

    if len(true_boundary_indices) == 0:
        region_labels[is_tumor] = "CT"
        region_labels[is_stroma] = "Stroma"
        return region_labels

    # STEP 2: Define zones
    boundary_coords = coords[true_boundary_indices]
    boundary_tree   = KDTree(boundary_coords)
    dists_to_boundary, _ = boundary_tree.query(coords)

    im_width_pixels = 5 * pixel_scale_100um
    is_proximal = dists_to_boundary <= im_width_pixels

    region_labels[is_proximal & (is_tumor | is_stroma)] = "IM"
    region_labels[is_tumor & (~is_proximal)] = "CT"
    region_labels[is_stroma & (~is_proximal)] = "Stroma"

    return region_labels


def classify_mets(row):
    any_mets = str(row["any_mets"]).strip().upper() in ["T", "TRUE", "1"]
    dist     = str(row["Distant_Mets"]).strip().upper() in ["T", "TRUE", "1"]
    if not any_mets: return "No_Mets"
    if dist: return "Distant_Mets"
    return "LN_Mets"


# ============================================================
# LOAD METADATA
# ============================================================
print("=" * 60)
print("Pseudobulk DE Pipeline — Dendritic Cells")
print("=" * 60)

meta = pd.read_csv(META_PATH)
meta["deident"] = meta["deident"].astype(str)
meta["met_group"] = meta.apply(classify_mets, axis=1)

visium_files = sorted(
    f for f in os.listdir(VISIUM_DIR)
    if f.endswith(".h5ad") and f.replace(".h5ad", "").isdigit()
)
print(f"Found {len(visium_files)} sample files\n")

# ============================================================
# STEP 1: Pseudobulk aggregation
# ============================================================
print("STEP 1: Pseudobulk aggregation")
print("-" * 40)

pseudobulk_counts = {}   # key = "patientID_region", value = count array
pseudobulk_meta = []     # metadata per pseudobulk sample
gene_names = None        # will be set from first sample

for i, f in enumerate(visium_files):
    sid = f.replace(".h5ad", "")
    print(f"  [{i+1}/{len(visium_files)}] {sid}...", end=" ")

    try:
        adata = sc.read_h5ad(os.path.join(VISIUM_DIR, f))

        if "histology" not in adata.obs.columns:
            print("SKIPPED (no histology)")
            continue

        # Store gene names from first sample
        if gene_names is None:
            gene_names = adata.var_names.tolist()
        else:
            # Ensure consistent gene ordering
            if adata.var_names.tolist() != gene_names:
                # Subset to shared genes
                shared = [g for g in gene_names if g in adata.var_names]
                if len(shared) < len(gene_names) * 0.9:
                    print(f"SKIPPED (too few shared genes: {len(shared)})")
                    continue
                gene_names = shared

        # Get raw counts
        X = adata[:, gene_names].X
        if sparse.issparse(X):
            X = X.toarray()

        # Cell2Location abundance
        ab = adata.obsm["means_cell_abundance_w_sf"]
        lineage_map = {col: map_to_h6(col) for col in ab.columns}
        h6_df = ab.groupby(lineage_map, axis=1).sum()

        # DC proportion per spot
        row_totals = h6_df.sum(axis=1)
        dc_prop = h6_df[TARGET_CELL_TYPE] / row_totals

        # Define regions
        coords = adata.obsm["spatial"]
        nbrs = NearestNeighbors(n_neighbors=2).fit(coords)
        distances, _ = nbrs.kneighbors(coords)
        pixel_scale_100um = np.median(distances[:, 1])
        region_labels = define_regions_spatial(adata, pixel_scale_100um)

        # Clinical metadata for this patient
        row = meta.loc[meta["deident"] == sid]
        if row.shape[0] != 1:
            print("SKIPPED (metadata mismatch)")
            continue
        clin = row.iloc[0]

        # Aggregate per region
        regions_found = []
        for region in ["CT", "IM", "Stroma"]:
            # DC-enriched spots in this region
            mask = (region_labels == region) & (dc_prop.values > DC_THRESHOLD)
            n_spots = mask.sum()

            if n_spots < MIN_SPOTS:
                continue

            # Sum raw counts across selected spots
            counts_sum = X[mask, :].sum(axis=0)
            # Ensure it's a 1D array
            counts_sum = np.asarray(counts_sum).flatten()

            sample_key = f"{sid}_{region}"
            pseudobulk_counts[sample_key] = counts_sum

            # QC metric: mean library size of included spots
            if sparse.issparse(adata.X):
                lib_sizes = np.asarray(adata.X[mask, :].sum(axis=1)).flatten()
            else:
                lib_sizes = adata.X[mask, :].sum(axis=1)
            mean_lib_size = np.log1p(np.mean(lib_sizes))

            pseudobulk_meta.append({
                "sample_key": sample_key,
                "patient_id": sid,
                "region": region,
                "mets_status": classify_mets(clin),
                "MLH1": int(clin["MLH1"]),
                "age": int(clin["age"]),
                "sex": clin["sex"],
                "n_spots": n_spots,
                "QC": mean_lib_size,
            })
            regions_found.append(f"{region}({n_spots})")

        if regions_found:
            print(f"OK — {', '.join(regions_found)}")
        else:
            print("SKIPPED (no region with enough DC spots)")

    except Exception as e:
        print(f"FAILED: {e}")

# ============================================================
# STEP 2: Build count matrix and metadata
# ============================================================
print(f"\nSTEP 2: Building count matrix")
print("-" * 40)

meta_df = pd.DataFrame(pseudobulk_meta).set_index("sample_key")
count_df = pd.DataFrame(pseudobulk_counts, index=gene_names).T

# Ensure alignment
count_df = count_df.loc[meta_df.index]

# Convert to integer (required by DESeq2)
count_df = count_df.astype(int)

print(f"  Pseudobulk samples: {count_df.shape[0]}")
print(f"  Genes: {count_df.shape[1]}")
print(f"\n  Region distribution:")
print(meta_df["region"].value_counts().to_string(header=False))
print(f"\n  Mets distribution:")
print(meta_df["mets_status"].value_counts().to_string(header=False))
print(f"\n  Region × Mets:")
print(pd.crosstab(meta_df["region"], meta_df["mets_status"]))

# Filter genes: keep only those with counts in enough samples
genes_detected = (count_df > 0).sum(axis=0)
keep_genes = genes_detected[genes_detected >= MIN_GENES_EXPR].index
count_df = count_df[keep_genes]
print(f"\n  Genes after filtering (detected in >= {MIN_GENES_EXPR} samples): {count_df.shape[1]}")

# Save intermediate files
count_df.to_csv(os.path.join(OUTPUT_DIR, "pseudobulk_counts.csv"))
meta_df.to_csv(os.path.join(OUTPUT_DIR, "pseudobulk_metadata.csv"))
print(f"  Saved to {OUTPUT_DIR}/")

# ============================================================
# STEP 3: Run pyDESeq2
# ============================================================
print(f"\nSTEP 3: Running pyDESeq2")
print("-" * 40)

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# --- Approach: Combined factor for clean contrasts ---
# Create a combined region_mets factor for easier contrast extraction
# This avoids interaction term complexity
meta_df["group"] = meta_df["region"] + "_" + meta_df["mets_status"]

print(f"\n  Groups:")
print(meta_df["group"].value_counts().to_string())

# Set categorical types
meta_df["group"] = pd.Categorical(meta_df["group"])
meta_df["sex"] = pd.Categorical(meta_df["sex"])
meta_df["MLH1"] = meta_df["MLH1"].astype(int)  # binary 0/1, treat as continuous

# --- Model A: Combined factor design ---
# ~ QC + MLH1 + age + sex + group
# This lets us pull out any pairwise comparison directly
print("\n--- Model A: Combined factor design ---")

dds = DeseqDataSet(
    counts=count_df,
    metadata=meta_df,
    design="~ QC + MLH1 + age + sex + group"
)
dds.deseq2()

# --- Define contrasts of interest ---
# Q1: Region effect (IM vs CT) within No_Mets
# Q2: Metastasis effect (Distant_Mets vs No_Mets) within IM
# Q3: Interaction — is the mets effect different in IM vs CT?

contrasts = {
    # Q1: Region effects within each mets group
    "IM_vs_CT__No_Mets": ["group", "IM_No_Mets", "CT_No_Mets"],
    "IM_vs_CT__LN_Mets": ["group", "IM_LN_Mets", "CT_LN_Mets"],
    "IM_vs_CT__Distant_Mets": ["group", "IM_Distant_Mets", "CT_Distant_Mets"],

    # Q2: Mets effects within each region
    "Distant_vs_No__CT": ["group", "CT_Distant_Mets", "CT_No_Mets"],
    "Distant_vs_No__IM": ["group", "IM_Distant_Mets", "IM_No_Mets"],
    "Distant_vs_No__Stroma": ["group", "Stroma_Distant_Mets", "Stroma_No_Mets"],
    "LN_vs_No__CT": ["group", "CT_LN_Mets", "CT_No_Mets"],
    "LN_vs_No__IM": ["group", "IM_LN_Mets", "IM_No_Mets"],
    "LN_vs_No__Stroma": ["group", "Stroma_LN_Mets", "Stroma_No_Mets"],

    # Stroma vs CT/IM comparisons
    "Stroma_vs_CT__No_Mets": ["group", "Stroma_No_Mets", "CT_No_Mets"],
    "Stroma_vs_IM__No_Mets": ["group", "Stroma_No_Mets", "IM_No_Mets"],
}

# Run each contrast
results_all = {}
summary_rows = []

for name, contrast in contrasts.items():
    # Check that both levels exist in the data
    if contrast[1] not in meta_df["group"].values or contrast[2] not in meta_df["group"].values:
        print(f"  SKIPPED {name}: one or both groups missing from data")
        continue

    try:
        stat_res = DeseqStats(dds, contrast=contrast, alpha=0.05)
        stat_res.summary()
        res_df = stat_res.results_df.copy()
        res_df = res_df.sort_values("padj")

        results_all[name] = res_df

        n_sig = (res_df["padj"] < 0.05).sum()
        n_up  = ((res_df["padj"] < 0.05) & (res_df["log2FoldChange"] > 0)).sum()
        n_down = ((res_df["padj"] < 0.05) & (res_df["log2FoldChange"] < 0)).sum()

        summary_rows.append({
            "contrast": name,
            "n_sig_005": n_sig,
            "n_up": n_up,
            "n_down": n_down,
            "top_gene": res_df.index[0] if n_sig > 0 else "none",
            "top_padj": res_df["padj"].iloc[0],
            "top_lfc": res_df["log2FoldChange"].iloc[0],
        })

        print(f"  {name}: {n_sig} DE genes (↑{n_up} ↓{n_down})")

        # Save full results
        res_df.to_csv(os.path.join(OUTPUT_DIR, f"DE_{name}.csv"))

    except Exception as e:
        print(f"  FAILED {name}: {e}")

# --- Model B: Additive + interaction design for global tests ---
print("\n--- Model B: Interaction design (region * mets_status) ---")

# Reset categorical for this model
meta_df["region"] = pd.Categorical(meta_df["region"], categories=["CT", "IM", "Stroma"])
meta_df["mets_status"] = pd.Categorical(
    meta_df["mets_status"], categories=["No_Mets", "LN_Mets", "Distant_Mets"]
)

try:
    dds_interaction = DeseqDataSet(
        counts=count_df,
        metadata=meta_df,
        design="~ QC + MLH1 + age + sex + region + mets_status + region:mets_status"
    )
    dds_interaction.deseq2()

    # Main effects
    for effect_name, contrast in [
        ("region_IM_vs_CT", ["region", "IM", "CT"]),
        ("region_Stroma_vs_CT", ["region", "Stroma", "CT"]),
        ("mets_LN_vs_No", ["mets_status", "LN_Mets", "No_Mets"]),
        ("mets_Distant_vs_No", ["mets_status", "Distant_Mets", "No_Mets"]),
    ]:
        try:
            stat_res = DeseqStats(dds_interaction, contrast=contrast, alpha=0.05)
            stat_res.summary()
            res_df = stat_res.results_df.sort_values("padj")
            n_sig = (res_df["padj"] < 0.05).sum()
            print(f"  {effect_name}: {n_sig} DE genes")
            res_df.to_csv(os.path.join(OUTPUT_DIR, f"DE_interaction_{effect_name}.csv"))
            results_all[f"interaction_{effect_name}"] = res_df
        except Exception as e:
            print(f"  FAILED {effect_name}: {e}")

except Exception as e:
    print(f"  Model B failed: {e}")
    print("  This can happen if some region×mets combinations have too few samples.")
    print("  Model A results (combined factor) are still valid.")

# ============================================================
# STEP 4: Summary and volcano plots
# ============================================================
print(f"\nSTEP 4: Summary and visualization")
print("-" * 40)

import matplotlib.pyplot as plt

# Summary table
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(OUTPUT_DIR, "DE_summary.csv"), index=False)
    print("\nDE Summary:")
    print(summary_df.to_string(index=False))

# Volcano plots for ALL contrasts with results
# Organized into logical groups

# Group 1: IM vs CT by mets status
# Group 2: Mets effect within each region
# Group 3: Stroma comparisons
volcano_groups = {
    "IM_vs_CT_by_mets": {
        "title": "IM vs CT — by Metastasis Status",
        "contrasts": ["IM_vs_CT__No_Mets", "IM_vs_CT__LN_Mets", "IM_vs_CT__Distant_Mets"],
    },
    "Distant_vs_No_by_region": {
        "title": "Distant Mets vs No Mets — by Region",
        "contrasts": ["Distant_vs_No__CT", "Distant_vs_No__IM", "Distant_vs_No__Stroma"],
    },
    "LN_vs_No_by_region": {
        "title": "LN Mets vs No Mets — by Region",
        "contrasts": ["LN_vs_No__CT", "LN_vs_No__IM", "LN_vs_No__Stroma"],
    },
    "Stroma_comparisons": {
        "title": "Stroma Comparisons — No Mets",
        "contrasts": ["Stroma_vs_CT__No_Mets", "Stroma_vs_IM__No_Mets"],
    },
}

def make_volcano(ax, res_df, title_str):
    """Draw a single volcano plot on the given axis."""
    res = res_df.dropna(subset=["padj", "log2FoldChange"])

    sig_up   = (res["padj"] < 0.05) & (res["log2FoldChange"] > 1)
    sig_down = (res["padj"] < 0.05) & (res["log2FoldChange"] < -1)
    ns       = ~(sig_up | sig_down)

    ax.scatter(res.loc[ns, "log2FoldChange"], -np.log10(res.loc[ns, "padj"]),
               c="gray", alpha=0.3, s=5)
    ax.scatter(res.loc[sig_up, "log2FoldChange"], -np.log10(res.loc[sig_up, "padj"]),
               c="#D62728", alpha=0.6, s=10, label=f"Up ({sig_up.sum()})")
    ax.scatter(res.loc[sig_down, "log2FoldChange"], -np.log10(res.loc[sig_down, "padj"]),
               c="#1F77B4", alpha=0.6, s=10, label=f"Down ({sig_down.sum()})")

    # Label top genes
    top_genes = res.head(10).index
    for gene in top_genes:
        if res.loc[gene, "padj"] < 0.05:
            ax.annotate(gene,
                        (res.loc[gene, "log2FoldChange"], -np.log10(res.loc[gene, "padj"])),
                        fontsize=5, alpha=0.8)

    ax.axhline(-np.log10(0.05), color="black", linestyle=":", alpha=0.3)
    ax.axvline(1, color="black", linestyle=":", alpha=0.3)
    ax.axvline(-1, color="black", linestyle=":", alpha=0.3)
    ax.set_xlabel("log2 Fold Change")
    ax.set_ylabel("-log10(padj)")
    ax.set_title(title_str)
    ax.legend(fontsize=8)

for group_key, group_info in volcano_groups.items():
    contrast_list = [c for c in group_info["contrasts"] if c in results_all]
    if len(contrast_list) == 0:
        continue

    ncols = len(contrast_list)
    fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))
    fig.suptitle(f"Pseudobulk DE — {TARGET_CELL_TYPE} (>{DC_THRESHOLD:.0%})\n{group_info['title']}",
                 fontsize=13, fontweight="bold")

    if ncols == 1:
        axes = [axes]

    for ax, cname in zip(axes, contrast_list):
        display_name = cname.replace("__", " | ").replace("_", " ")
        make_volcano(ax, results_all[cname], display_name)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"volcano_{group_key}.png"), dpi=150, bbox_inches="tight")
    plt.savefig(os.path.join(OUTPUT_DIR, f"volcano_{group_key}.pdf"), bbox_inches="tight")
    plt.close()
    print(f"  Saved volcano_{group_key}.pdf")

print("All volcano plots saved")

# ============================================================
# DONE
# ============================================================
print(f"\n{'=' * 60}")
print("COMPLETE")
print(f"{'=' * 60}")
print(f"Results saved to: {OUTPUT_DIR}/")
print(f"  - pseudobulk_counts.csv: aggregated count matrix")
print(f"  - pseudobulk_metadata.csv: sample metadata")
print(f"  - DE_*.csv: full DE results per contrast")
print(f"  - DE_summary.csv: summary table")
print(f"  - volcano_plots.pdf: volcano plots for key contrasts")
print(f"\nDesign: ~ QC + MLH1 + age + sex + group")
print(f"  where group = region × mets_status (combined factor)")
print(f"  DC threshold: {DC_THRESHOLD:.0%}")
print(f"  Min spots per pseudobulk sample: {MIN_SPOTS}")

Pseudobulk DE Pipeline — Dendritic Cells
Found 41 sample files

STEP 1: Pseudobulk aggregation
----------------------------------------
  [1/41] 0... OK — CT(2874), IM(1307), Stroma(2080)
  [2/41] 10... OK — CT(3310), IM(582), Stroma(1776)
  [3/41] 100... OK — CT(6841), IM(168)
  [4/41] 102... OK — CT(356), Stroma(2914)
  [5/41] 106... OK — CT(4234), IM(1098), Stroma(94)
  [6/41] 107... OK — CT(3205), IM(363), Stroma(2861)
  [7/41] 11... OK — CT(4683), IM(884), Stroma(68)
  [8/41] 111... OK — CT(4669), IM(1064), Stroma(835)
  [9/41] 116... OK — CT(6093), IM(965), Stroma(83)
  [10/41] 117... OK — CT(4394), IM(126)
  [11/41] 119... OK — CT(1312), IM(1001), Stroma(50)
  [12/41] 122... OK — CT(3579), IM(998)
  [13/41] 127... OK — CT(6270), IM(376), Stroma(90)
  [14/41] 128... OK — CT(4830)
  [15/41] 13... OK — CT(2938), IM(783), Stroma(226)
  [16/41] 14... OK — CT(2604), IM(2618), Stroma(166)
  [17/41] 18... OK — CT(1640), IM(1647), Stroma(78)
  [18/41] 28... OK — CT(6061), IM(358)
  [19/4

Fitting size factors...
... done in 0.03 seconds.

/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:378: RuntimeWarning: overflow encountered in exp
  mu_ = np.maximum(size_factors * np.exp(X @ beta), min_mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:230: RuntimeWarning: invalid value encountered in subtract
  -logbinom
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:382: RuntimeWarning: overflow encountered in exp
  mu_ = np.maximum(size_factors * np.exp(X @ beta), min_mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:385: RuntimeWarning: invalid value encountered in divide
  + ((1 / disp + counts) * mu_ / (1 / disp + mu_)) @ X
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/utils.py:378: RuntimeWarning: overflow encountered in exp
  mu_ = np.maximum(size_factors * np.exp(X @ beta), min_mu)
/opt/anaconda3/envs/visium/lib/python3.10/site-packages/pydeseq2/util

Log2 fold change & Wald test p-value: group IM_No_Mets vs CT_No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
SAMD11   178.896210        0.844936  0.206522  4.091255  0.000043  0.004444
NOC2L    567.576507       -0.158526  0.128285 -1.235729  0.216559  0.539830
KLHL17   189.625772        0.103484  0.190728  0.542577  0.587421  0.812288
PLEKHN1  182.574237       -0.037357  0.293563 -0.127253  0.898740  0.962348
HES4     919.920090        0.156509  0.237691  0.658459  0.510244  0.768264
...             ...             ...       ...       ...       ...       ...
VBP1     539.104697       -0.192096  0.146175 -1.314151  0.188795  0.509783
CLIC2    232.988784        0.674891  0.226246  2.982999  0.002854  0.049825
TMLHE    472.286222       -0.149322  0.138038 -1.081742  0.279367  0.597654
VAMP7    673.498409       -0.077672  0.108051 -0.718850  0.472233  0.743645
IL9R      34.975878        0.496500  0.243698  2.037357  0.041614  0.261665

[13598 rows x 6 co

... done in 0.36 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: group IM_LN_Mets vs CT_LN_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
SAMD11   178.896210        0.840124  0.246151  3.413041  0.000642  0.042147
NOC2L    567.576507       -0.314535  0.153373 -2.050781  0.040288  0.196647
KLHL17   189.625772        0.068395  0.227392  0.300782  0.763581  0.875760
PLEKHN1  182.574237       -0.147482  0.350774 -0.420448  0.674158  0.820634
HES4     919.920090        0.014232  0.284506  0.050024  0.960103  0.984202
...             ...             ...       ...       ...       ...       ...
VBP1     539.104697       -0.410924  0.174918 -2.349236  0.018812  0.138127
CLIC2    232.988784        0.780686  0.269236  2.899633  0.003736  0.072219
TMLHE    472.286222       -0.188316  0.165543 -1.137565  0.255302  0.486314
VAMP7    673.498409       -0.177974  0.129087 -1.378713  0.167983  0.387591
IL9R      34.975878        0.490042  0.286609  1.709794  0.087304       NaN

[13598 rows x 6 co

...

Log2 fold change & Wald test p-value: region Stroma vs CT
           baseMean  log2FoldChange     lfcSE      stat        pvalue  \
SAMD11   178.896210        1.398931  0.241264  5.798351  6.697036e-09   
NOC2L    567.576507       -0.501670  0.152189 -3.296357  9.794752e-04   
KLHL17   189.625772       -0.200166  0.226831 -0.882446  3.775359e-01   
PLEKHN1  182.574237       -0.643330  0.344301 -1.868510  6.169107e-02   
HES4     919.920090        0.102252  0.271585  0.376502  7.065441e-01   
...             ...             ...       ...       ...           ...   
VBP1     539.104697       -0.408185  0.172791 -2.362310  1.816143e-02   
CLIC2    232.988784        1.398682  0.261424  5.350247  8.783418e-08   
TMLHE    472.286222       -0.241580  0.163650 -1.476193  1.398920e-01   
VAMP7    673.498409        0.058957  0.127823  0.461244  6.446235e-01   
IL9R      34.975878        0.597203  0.300284  1.988794  4.672395e-02   

                 padj  
SAMD11   2.702264e-07  
NOC2L    4.120420

... done in 0.37 seconds.

Running Wald tests...


Log2 fold change & Wald test p-value: mets_status LN_Mets vs No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
SAMD11   178.896210        0.379894  0.206281  1.841636  0.065528  0.372515
NOC2L    567.576507       -0.075716  0.127868 -0.592143  0.553755  0.831948
KLHL17   189.625772       -0.071047  0.189507 -0.374907  0.707730  0.905420
PLEKHN1  182.574237        0.032956  0.294432  0.111930  0.910879  0.980722
HES4     919.920090        0.186137  0.240226  0.774840  0.438434  0.758985
...             ...             ...       ...       ...       ...       ...
VBP1     539.104697       -0.008946  0.145947 -0.061296  0.951123  0.987889
CLIC2    232.988784        0.328263  0.226426  1.449759  0.147126  0.496431
TMLHE    472.286222       -0.013092  0.137859 -0.094965  0.924343  0.984384
VAMP7    673.498409        0.152617  0.107491  1.419814  0.155662  0.509800
IL9R      34.975878        0.360704  0.234003  1.541448  0.123208  0.464363

[13598 rows x 6 co

... done in 0.36 seconds.



Log2 fold change & Wald test p-value: mets_status Distant_Mets vs No_Mets
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
SAMD11   178.896210        0.444918  0.217695  2.043772  0.040976  0.335125
NOC2L    567.576507        0.126508  0.135562  0.933209  0.350712  0.729436
KLHL17   189.625772       -0.294668  0.200914 -1.466640  0.142474  0.526887
PLEKHN1  182.574237        0.016608  0.312656  0.053119  0.957637  0.989961
HES4     919.920090       -0.079543  0.255331 -0.311528  0.755399  0.927487
...             ...             ...       ...       ...       ...       ...
VBP1     539.104697        0.039295  0.154763  0.253906  0.799568  0.939913
CLIC2    232.988784       -0.063504  0.240104 -0.264485  0.791406  0.937253
TMLHE    472.286222       -0.054414  0.145915 -0.372917  0.709210  0.910464
VAMP7    673.498409        0.218006  0.113791  1.915835  0.055386  0.367256
IL9R      34.975878       -0.130120  0.246767 -0.527298  0.597987  0.861917

[13598 rows x